In [ ]:
import numpy as np

In [ ]:
import cupy as cp

In [ ]:
!nvidia-smi

# `cuda.compute`: Parallel Computing Primitives

The `cuda.compute` library provides composable primitives for building custom parallel algorithms on the GPU—without writing CUDA kernels directly.

https://nvidia.github.io/cccl/unstable/python/compute/index.html



In [1]:
import cuda.compute

In [ ]:
"""
Sum all values in an array using reduction with PLUS operation.
"""
from cuda.compute import OpKind

# Prepare the input and output arrays.
dtype = np.int32
h_init = np.array([0], dtype=dtype)
d_input = cp.array([1, 2, 3, 4, 5], dtype=dtype)
d_output = cp.empty(1, dtype=dtype)

# Perform the reduction.
cuda.compute.reduce_into(
    d_in=d_input, d_out=d_output, num_items=len(d_input), op=OpKind.PLUS, h_init=h_init
)

# Verify the result.
expected_output = 15
assert (d_output == expected_output).all()
result = d_output[0]
print(f"Sum reduction result: {result}")

## Stream-Based Computing with `cuda.compute` Iterators

When designing high-performance parallel pipelines on GPUs, **memory bandwidth is often the primary bottleneck**. Traditional array-processing frameworks (such as NumPy or a naïve sequence of CuPy operations) execute one operation at a time across the entire dataset, materializing temporary intermediate arrays in GPU memory (VRAM) between each step.

For example, suppose you want to square every element of an array and then compute their sum. A traditional pipeline first launches a kernel to compute the squares and stores the results in a temporary array. A second kernel then reads that temporary array to perform the reduction. For large datasets, repeatedly allocating, writing, and reading intermediate arrays increases memory traffic and reduces performance.

To address this, `cuda.compute` provides a **lazy, stream-based programming model** built around **iterators**.

```text
Traditional pipeline (materializes intermediate arrays)

Input Array
     │
     ▼
Square Kernel
     │
     ▼
Temporary Array (x²)
     │
     ▼
Reduction Kernel
     │
     ▼
Result


Iterator-based pipeline (no intermediate array)

Input Array
     │
     ▼
TransformIterator (x²)
     │
     ▼
Reduction Kernel
     │
     ▼
Result
```

Rather than materializing intermediate arrays, iterators represent **logical sequences whose elements are computed on demand**. They describe *how* values should be produced instead of *where* they are stored.

This allows transformations, generated sequences, and structured views of data to be fused directly into GPU algorithms. Parallel primitives such as `reduce_into` and `scan_into` consume values from an iterator as needed, eliminating unnecessary intermediate storage while reducing memory traffic and improving performance.

`cuda.compute` currently provides four iterator types:

* **`CountingIterator`** generates arithmetic sequences on demand without allocating an input array.
* **`TransformIterator`** lazily applies the same function to every input element, enabling operation fusion.
* **`ZipIterator`** combines multiple arrays or iterators into a single logical stream of tuples without copying data.
* **`TransformOutputIterator`** applies a transformation to the result of an algorithm immediately before writing it to the output array.

Together, these iterators enable GPU algorithms to operate on **streams of values rather than materialized arrays**, making it possible to express complex data-processing pipelines while minimizing memory movement.

<img src="img/iterators.png" alt="Iterators"/>

## Composable GPU Primitives with `cuda.compute`

This interactive tutorial introduces `cuda.compute`, a library of high-performance GPU primitives for parallel computing without requiring users to write CUDA kernels manually. Under the hood, these primitives leverage Numba CUDA to just-in-time (JIT) compile Python functions into efficient GPU device code.

## Core Architecture and Conventions

Before tackling the puzzles, it is helpful to understand the three core API conventions used throughout `cuda.compute`.

### Keyword-only arguments

All algorithm parameters must be passed explicitly by name. Passing positional arguments raises a `TypeError`.

```python
# ❌ Incorrect
cuda.compute.reduce_into(d_in, d_out, n, OpKind.PLUS, h_init)

# ✅ Correct
cuda.compute.reduce_into(
    d_in=d_in,
    d_out=d_out,
    num_items=n,
    op=OpKind.PLUS,
    h_init=h_init,
)
```

### Memory prefixes

Variable names indicate where data resides.

* `d_` denotes data in **device memory** (for example, CuPy arrays).
* `h_` denotes data in **host memory** (for example, NumPy arrays or scalar containers).

These prefixes make data movement explicit and improve code readability.

### Explicit output semantics

Algorithms write their results directly into preallocated output arrays instead of allocating and returning new arrays. This design minimizes memory allocations, gives users explicit control over GPU memory, and enables efficient composition of multiple GPU operations.


### Puzzle 1: Fused Operation Streaming (5-Minute Estimate)
#### Objective
Given an array containing integers, compute the sum of squares of only the odd values in a single pass across the data without allocating intermediate arrays for the transformations.

#### 💡 Hint
You can fuse transformations directly into the reduction pipeline using a `TransformIterator`. It lazily applies a unary lambda operation on demand as data passes into the reduction framework.

In [ ]:
from cuda.compute import OpKind, TransformIterator

# Target data sequence
h_data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.int32)
d_input = cp.array(h_data)

# Pre-allocated single-element destination buffer on device
d_output = cp.empty(1, dtype=np.int32)

# Host initialization container for the accumulator
h_init = np.array([0], dtype=np.int32)

#### Code Challenge
Complete the implementation below to construct a lazy mapping stream and reduce it into `d_output`:

```python
# 1. Instantiate a TransformIterator to lazily evaluate: 
#    x**2 if x is odd, else 0
it_input = TransformIterator(
    d_input, 
    lambda x: # ### YOUR CODE HERE ###
)

# 2. Execute the reduction using explicit keyword-only arguments
cuda.compute.reduce_into(
    # ### YOUR CODE HERE ###
)

# Validate results
expected_sum = sum(x**2 for x in h_data if x % 2 != 0)
assert int(d_output[0]) == expected_sum
print(f"✅ Success! Fused sum of odd squares: {int(d_output[0])}")

<details>
<summary><b>Click to reveal the 5-minute challenge: "Fused Operation Streaming"</b></summary>

```python
it_input = TransformIterator(d_input, lambda x: x**2 if x % 2 != 0 else 0)

cuda.compute.reduce_into(
    d_in=it_input,
    d_out=d_output,
    num_items=len(d_input),
    op=OpKind.PLUS,
    h_init=h_init
)

## Puzzle 2: Parallel Argmin Lookup via Tuples (10-Minute Estimate)
### Objective
Locate both the **minimum value** and its associated **index** inside an array using a single parallel passes across a Structure of Arrays (SoA) layout.  
### 💡 Hint
You can group disjoint parallel streams together horizontally using a `ZipIterator`. When pairing a sequential `CountingIterator` with an array, the zipper streams logical structural tuple elements (`(index, value)`) down to your customized device reduction function.  Because the reduction handles structured pairs, the initialization state (`h_init`) and target buffer (`d_output`) must match a matching structured data type. 

In [ ]:
from cuda.compute import CountingIterator, ZipIterator

# Target dataset array containing an explicit global minimum value
h_dataset = np.array([42, 17, 89, 5, 66, 12, 91, 4], dtype=np.int32)
d_dataset = cp.array(h_dataset)

# Configure the tracking structured data type
pair_dtype = np.dtype([("index", np.int32), ("value", np.int32)], align=True)

# Allocate host and device metadata tracking boundaries
h_init = np.asarray([(-1, np.iinfo(np.int32).max)], dtype=pair_dtype)
d_output = cp.empty(1, dtype=pair_dtype)

#### Code Challenge
Write the custom binary comparison operation and compile the stream zipper sequence:

```python
# 1. Define the JIT-compiled reduction logic tracking the absolute minimum value
def min_by_value(p1, p2):
    """
    Evaluates two structured tuple records and passes onwards 
    the one containing the smaller value element.
    """
    # ### YOUR CODE HERE ###


# 2. Instantiate a sequential coordinate position counter starting at index 0
counting_it = # ### YOUR CODE HERE ###


# 3. Zip coordinate positions together with physical target metrics
zip_it = # ### YOUR CODE HERE ###


# 4. Invoke the reduction pipeline using mandatory keyword configurations
cuda.compute.reduce_into(
    # ### YOUR CODE HERE ###
)

# Validate correctness
result = d_output.get()[0]
expected_idx = int(np.argmin(h_dataset))
expected_val = int(h_dataset[expected_idx])

assert result["index"] == expected_idx and result["value"] == expected_val
print(f"✅ Success! Global Argmin Located at Index [{result['index']}] -> Value: {result['value']}")

<details>
<summary><b>Click to reveal the 10-minute challenge: "Parallel Argmin Lookup via Tuples"</b></summary>

```python
def min_by_value(p1, p2):
    return p1 if p1[1] < p2[1] else p2

counting_it = CountingIterator(np.int32(0))
zip_it = ZipIterator(counting_it, d_dataset)

cuda.compute.reduce_into(
    d_in=zip_it,
    d_out=d_output,
    num_items=len(d_dataset),
    op=min_by_value,
    h_init=h_init
)